# Track 1 image-conditioned object shape
Generate a normalized geometry hypothesis from the automatically segmented video.
This is not yet a metric reconstruction. Keep external model licenses separate
from this repository's Apache license. No Track 2 assets or reference poses.

In [ ]:
import os,sys,subprocess,json
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run(['uv','python','install','3.11'],check=True)
subprocess.run(['uv','venv','--python','3.11','/tmp/shape-env'],check=True)
py='/tmp/shape-env/bin/python'
subprocess.run(['uv','pip','install','--python',py,'torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu124'],check=True)
subprocess.run(['uv','pip','install','--python',py,'transformers==4.48.3','diffusers==0.32.2','accelerate','einops','omegaconf','trimesh','pillow','numpy<2','scikit-image','opencv-python<4.12','pyyaml','tqdm','safetensors','huggingface_hub<1','pygltflib','pymeshlab','fast_simplification','ninja','pybind11'],check=True)
repo=Path('/tmp/Hunyuan3D-2')
subprocess.run(['git','clone','--depth','1','https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git',str(repo)],check=True)
revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
subprocess.run(['uv','pip','install','--python',py,'--no-deps','-e',str(repo)],check=True)
env=os.environ.copy();env['PYTHONPATH']=str(repo)
images=list(Path('/kaggle/input').rglob('object_observation.png'))
assert len(images)==1,images


In [ ]:
Path('/tmp/reconstruct_object.py').write_text('"""Reconstruct an object from its automatic RGBA video crop using Hunyuan3D.\n\nThis produces a normalized shape hypothesis. Metric scale and camera registration\nmust be estimated from the Track 1 observations before using it in a submission.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom huggingface_hub import HfApi, snapshot_download\nfrom hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--image\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--weights\',type=Path,required=True)\n    p.add_argument(\'--revision\',help=\'Pin a previously recorded HF model revision\')\n    p.add_argument(\'--steps\',type=int,default=50)\n    p.add_argument(\'--resolution\',type=int,default=384)\n    p.add_argument(\'--seed\',type=int,default=42)\n    a=p.parse_args()\n    if a.steps<1 or a.resolution<64:p.error(\'Invalid sampling settings\')\n    image=Image.open(a.image).convert(\'RGBA\')\n    alpha=np.asarray(image)[:,:,3]\n    if not (alpha>0).any() or (alpha>0).all():\n        raise ValueError(\'Expected an actual masked RGBA observation\')\n    repo=\'tencent/Hunyuan3D-2\'\n    revision=a.revision or HfApi().model_info(repo).sha\n    snapshot_download(repo,revision=revision,local_dir=a.weights,\n        allow_patterns=[\'hunyuan3d-dit-v2-0/**\',\'hunyuan3d-vae-v2-0/**\',\'*.json\'])\n    pipeline=Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(str(a.weights),\n        subfolder=\'hunyuan3d-dit-v2-0\',device=\'cuda\',dtype=torch.float16)\n    with torch.inference_mode():\n        mesh=pipeline(image=image,num_inference_steps=a.steps,\n            octree_resolution=a.resolution,num_chunks=4000,\n            generator=torch.Generator(device=\'cuda\').manual_seed(a.seed))[0]\n    if not len(mesh.vertices) or not len(mesh.faces) or not np.isfinite(mesh.vertices).all():\n        raise ValueError(\'Image-conditioned model produced invalid geometry\')\n    a.output.mkdir(parents=True,exist_ok=True)\n    mesh.export(a.output/\'normalized_shape.glb\')\n    report={\'model\':repo,\'revision\':revision,\'seed\':a.seed,\'steps\':a.steps,\n        \'resolution\':a.resolution,\'input_sha256\':hashlib.sha256(a.image.read_bytes()).hexdigest(),\n        \'vertices\':len(mesh.vertices),\'faces\':len(mesh.faces),\n        \'metric_scale_estimated\':False,\'camera_registration_estimated\':False,\n        \'ready_for_submission\':False,\'manual_geometry_used\':False}\n    (a.output/\'shape_report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n')
subprocess.run([py,'/tmp/reconstruct_object.py','--image',str(images[0]),'--output','/kaggle/working/episode_000000','--weights','/tmp/shape_weights'],env=env,check=True)
Path('/kaggle/working/code_revision.json').write_text(json.dumps({'Hunyuan3D-2':revision}))